# SSLimPy — Higher-order Statistics: PT Kernels and the Trispectrum

The non-Gaussian covariance of the power spectrum is built from the 4-point
function of the line-emission field. Its kernels live in
`SSLimPy/LIMsurvey/higher_order.py` (standard perturbation theory) and
`SSLimPy/LIMsurvey/ingredients_T0.py` (the analytic no-radial-canonical
expressions used for the 1-, 2-, 3- and 4-halo terms). This notebook evaluates
these ingredients directly and builds the halo trispectrum for a [CII] survey.

In [ ]:
import os
import sys

sys.path.append("../")

# Use the available cores for the Einstein--Boltzmann solver (CAMB/CLASS) and numba
os.environ.setdefault("OMP_NUM_THREADS", "12")

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

Cs = sns.color_palette("colorblind")
import niceplots
niceplots.utils.initPlot()

## 1. Setup

In [ ]:
settings = {
    "code": "camb",
    "do_RSD": True,
    "nonlinearRSD": True,
    "QNLpowerspectrum": True,
    "halo_model_PS": True,
    "output": ["Power spectrum"],
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 10 * u.Mpc**-1,
    "nk": 32,
}

cosmodict = {
    "h": 0.677,
    "Omegam": 0.309177,
    "Omegab": 0.04903,
    "sigma8": 0.8222,
    "ns": 0.96824,
    "mnu": 0.06,
    "Neff": 3.044,
}

halodict = {
    "halo_tracer": "clustering",
    "hmf_model": "ST",
    "concentration": "Diemer19",
    "bias_model": "ST99",
    "bias_par": {"q": 0.707, "p": 0.3},
}

astrodict = {
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {"a": 0.8475, "b": 7.2203, "SFR_file": "sfr_release.dat", "do_quench": False},
    "sigma_scatter": 0.37,
    "meanperserve_scatter": True,
}

surveyspecs = {
    "Tsys_NEFD": 40 * u.uK,
    "Nfeeds": 19,
    "tobs": 1300 * u.h,
    "nD": 1,
    "beam_FWHM": 0.5 * u.arcmin,
    "nu": 1.897 * u.THz,
    "nuObs": 250 * u.GHz,
    "Delta_nu": 50 * u.GHz,
    "dnu": 900 * u.MHz,
    "Omega_field": 140 * u.deg**2,
}

In [ ]:
from SSLimPy.interface import sslimpy

myssl = sslimpy.SSLimPy(
    settings_dict=settings, cosmopars=cosmodict, halopars=halodict,
    astropars=astrodict, obspars_dict=surveyspecs,
)

pobs = myssl.compute(
    myssl.fiducialcosmoparams, myssl.fiducialhaloparams,
    myssl.fiducialastroparams, myssl.fiducialspecparams,
    pobs_settings={"kmin": 1e-3 * u.Mpc**-1, "kmax": 1 * u.Mpc**-1, "nk": 20, "nmu": 32},
    output=["Power spectrum"],
)["Power spectrum"]

k = pobs.k

## 2. Standard perturbation-theory kernels

The kernels are numba-compiled scalar functions of the wave-vector geometry
$(k_i, \mu_i, \phi_i)$, with $\mu$ the angle to the line of sight.

In [ ]:
from SSLimPy.LIMsurvey import higher_order as kho

k1, k2 = 0.10, 0.22            # wave-vector magnitudes [Mpc^-1]
mu1, mu2 = 0.4, -0.6           # line-of-sight projections
phi1, phi2 = 0.3, 1.1          # azimuthal angles

print("vF2  =", kho.vF2(k1, mu1, phi1, k2, mu2, phi2))
print("vG2  =", kho.vG2(k1, mu1, phi1, k2, mu2, phi2))
print("alpha=", kho.alpha(k1, mu1, phi1, k2, mu2, phi2))
print("beta =", kho.beta(k1, mu1, phi1, k2, mu2, phi2))
print("Galileon2=", kho.Galileon2(k1, mu1, phi1, k2, mu2, phi2))
print("Galileon3=", kho.Galileon3(k1, mu1, phi1, k2, mu2, phi2, k1, mu1, 0.0))

The kernel $\mathcal{Z}_1,\mathcal{Z}_2$ (RSD kernels of the biased tracer,
with linear RSD parameter $f$ and eulerian biases):

In [ ]:
f = 0.9
b1 = 1.1
b2 = -0.3
bG2 = 0.1

mu1 = np.linspace(-1, 1, 200)

color = iter(Cs)
Z1 = [kho.vZ1(b1, f, 0, mui, phi1) for mui in mu1]
plt.plot(mu1, Z1, label=r"$\mathcal{Z}_1$", c=next(color))

Z2 = [kho.vZ2(b1, b2, bG2, f, k1, mui, phi1, k2, mu2, phi2) for mui in mu1]
plt.plot(mu1, Z2, label=r"$\mathcal{Z}_2$", c=next(color))

plt.legend()
plt.xlabel(r"$\mu_1$")
plt.ylabel("SPT kernel")

## 3. Analytic trispectrum ingredients

The (squeezed) 4-halo and 2-halo pieces use the analytic kernels of
`ingredients_T0.py`. They expect $k_1,k_2$ as `Quantity` (leaving the units
inside) and the (scale-dependent) bias parameters as arrays/numbers —
e.g. $T_{3111}(k_1,k_2) \propto \langle T b \rangle^4 P(k_1)P(k_2)$ for the
constant-bias case below

In [ ]:
from SSLimPy.utils import fft_log

k = myssl.current_cosmology.k
Pk_lin = myssl.current_cosmology.matpow

myfflog = fft_log.FFTLog(
    Pk_lin, k.min() / 5, k.max() * 5, 6, dict(z=0, nonlinear=False, tracer="clustering")
)
gamma, coef = myfflog.get_power_and_coef()

In [ ]:
plt.semilogx(k, 100 * (myfflog(k)/ Pk_lin(k, 0, False, "clustering").value - 1), c=Cs[0])
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"rel. deviation $[\%]$")

In [ ]:
from SSLimPy.LIMsurvey import ingredients_T0

T3111 = ingredients_T0.T3111_squeezed(k, 1, 0, 0, 0, 0, 0, 0) * Pk_lin(k, 0, False, "clustering")
T2211_A = np.zeros_like(T3111.value)
for coefi, gammai in zip(coef, gamma):
    T2211_A += (coefi * ingredients_T0.T2211_A_squeezed(k, 1, 0, 0, gammai)).real
T2211_A = T2211_A * u.Mpc**3

T2211_X = np.zeros_like(T3111.value)
for coefi, gammai in zip(coef, gamma):
    T2211_X += (coefi * ingredients_T0.T2211_X_squeezed(k, 1, 0, 0, gammai)).real
T2211_X = T2211_X * u.Mpc**3

In [ ]:
plt.loglog(k, -T3111, ls="--", label=r"$T_{31}$", c=Cs[0])
plt.loglog(k, T2211_A, label=r"$T_{22}$", c=Cs[1])
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$T_i \, P^{-2}\,[\mathrm{Mpc}^3]$")


## 4. Halo trispectrum of the line-emission field

The `nonGuassianCov` class wraps the survey-dependent versions. 1-halo and
2-halo terms are the dominant ones at large and small scales, respectively; the
3-halo term is small and the 4-halo term is largest at quasi-linear scales.

In [ ]:
from SSLimPy.LIMsurvey import covariance as scov

nG = scov.nonGuassianCov(pobs)
k = nG.k

T_1h = nG.integrate_1h().squeeze()
T_2h = nG.integrate_2h().squeeze().to(T_1h.unit)
T_3h = nG.integrate_3h().squeeze().to(T_1h.unit)
T_4h = nG.integrate_4h().squeeze().to(T_1h.unit)

color = iter(Cs)
for T, lab in zip([T_1h, T_2h, T_3h, T_4h], ["1-halo", "2-halo", "3-halo", "4-halo"]):
    plt.loglog(k, np.diag(T), label=lab, c=next(color))

plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$T(k,k)\,[\mu\mathrm{K}^4\,\mathrm{Mpc}^{9}]$")
plt.legend()

## 5. Normalised diagonal $\mathrm{Cov}(k,k)\,P^{-2}(k)$

The (squared-velocity / shot-noise-normalised) ratio tells us the effective
"non-Gaussianity" of the covariance at each $k$.

In [ ]:
Pk0 = pobs.Pk_0bs.squeeze().to(u.uK**2 * u.Mpc**3).value

color = iter(Cs)
for T in [T_1h, T_2h, T_3h, T_4h]:
    diag = np.diag(T).value
    plt.loglog(k, np.abs(diag) / Pk0**2, c=next(color))

plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}_{\rm PT}(k,k)\,P^{-2}(k)$ [arb. units]")